# D1 — BOS-attention diagnostic: Pythia-12B late-layer binding resurgence

Pre-registration: docs/d1-preregistration.md (COMMIT BEFORE RUNNING).
Prediction P1: ≥ half of late-layer (layer ≥ 27) top-binding heads at 12B are
sink-dominated (mean BOS attention ≥ 0.5) or structural (mean attention to
position 1 ≥ 0.5). All branches ship.

Colab: Runtime → A100 / high-RAM GPU. Run cells top to bottom.

In [1]:
import os

DEVELOPMENT_MODE = False

try:
    import google.colab
    IN_COLAB = True
    print("Running as a Colab notebook")
    os.system('pip install transformer_lens==2.17.0 --quiet')
    os.system('pip install transformers==4.44.0 --quiet')
    os.system('pip install --upgrade numpy --quiet')
    os.system('scipy --upgrade --quiet')
    print("DONE — now go to Runtime → Restart session, then skip this cell")
except ImportError:
    IN_COLAB = False

Running as a Colab notebook
DONE — now go to Runtime → Restart session, then skip this cell


In [1]:
import sys
import torch
import pandas as pd
import numpy as np
from pathlib import Path

# Colab: mount Drive and use My Drive/tmlr-results as root
# Local: notebook is in notebooks/, project root is one level up
if 'google.colab' in sys.modules:
    PROJECT_ROOT = Path('/content')
else:
    PROJECT_ROOT = Path.cwd().parent
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /content


In [6]:
from src.head_characterization import (
    get_top_binding_heads,
    characterize_heads,
    SINK_THRESHOLD,
    STRUCTURAL_THRESHOLD,
)

BINDING_CSV = PROJECT_ROOT / "results/pythia/pythia-12b-binding.csv"
OUT_DIR = PROJECT_ROOT / "results/d1_bos_diagnostic"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LATE_LAYER_MIN = 27  # frozen: final quarter of 36 layers (see pre-registration)

In [9]:
heads = get_top_binding_heads(BINDING_CSV)
print(f"Top binding heads from frozen CSV: {heads}")

late_heads = [(int(row['layer']), int(row['head'])) for _, row in heads.iterrows() if row['layer'] >= LATE_LAYER_MIN]
assert late_heads, "No late-layer heads in top set — record this outcome; it is itself reportable."

Top binding heads from frozen CSV:    layer  head  mean_binding  max_binding  n_compounds_top  \
0     20    33        0.9969       0.9969                1   
1      1    26        0.9693       0.9880                2   
2      3    30        0.9382       0.9963                3   
3      1    15        0.9010       0.9905                2   
4      5    28        0.7098       0.9910                1   
5     34    36        0.6510       0.9982                1   
6      3     9        0.3686       0.9167                1   

                              top_compounds  
0                                 link_text  
1                color_contrast, page_title  
2  alt_text, keyboard_navigation, skip_link  
3                 form_label, screen_reader  
4                           focus_indicator  
5                             semantic_html  
6                           closed_captions  


In [10]:
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained(
    "pythia-12b",
    torch_dtype=torch.bfloat16,
    device="cuda",
)
model.eval()
print(model.cfg.n_layers, "layers,", model.cfg.n_heads, "heads")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/9.81G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/9.93G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Loaded pretrained model pythia-12b into HookedTransformer
36 layers, 40 heads


In [12]:
# characterize_heads doesn't compute BOS/pos-1 attention, so pull directly from cache
prompt = "The alt text"  # use your standard prompt
tokens = model.to_tokens(prompt)
_, cache = model.run_with_cache(tokens)

results = []
for l, h in late_heads:
    attn = cache[f"blocks.{l}.attn.hook_pattern"][0, h]
    bos_attn = attn[:, 0].mean().item()
    pos1_attn = attn[:, 1].mean().item()
    results.append({"layer": l, "head": h, "bos_attn": bos_attn, "pos1_attn": pos1_attn})

table = pd.DataFrame(results)
print(table)

COL_BOS = "bos_attn"
COL_POS1 = "pos1_attn"
print(f"Using columns: BOS={COL_BOS}, POS1={COL_POS1}")

   layer  head  bos_attn  pos1_attn
0     34    36  0.957188    0.00455
Using columns: BOS=bos_attn, POS1=pos1_attn


In [13]:
table["sink_dominated"] = table[COL_BOS] >= SINK_THRESHOLD
table["structural"] = (table[COL_POS1] >= STRUCTURAL_THRESHOLD) if COL_POS1 else False
table["qualifies"] = table["sink_dominated"] | table["structural"]

n_late = len(table)
n_qual = int(table["qualifies"].sum())
frac = n_qual / n_late

if n_qual == 0:
    verdict = "NOT CONFIRMED"
elif frac >= 0.5:
    verdict = "CONFIRMED"
else:
    verdict = "PARTIAL"

print(f"\nD1 VERDICT: {verdict} — {n_qual}/{n_late} late-layer binders sink-or-structural (frozen bar: ≥ 50%)")


D1 VERDICT: CONFIRMED — 1/1 late-layer binders sink-or-structural (frozen bar: ≥ 50%)


In [14]:
table.to_csv(OUT_DIR / "d1_late_head_characterization.csv", index=True)

verdict_md = f"""# D1 VERDICT — {verdict}

Pre-registration: docs/d1-preregistration.md (frozen before run).
Head set (derived from results/pythia/pythia-12b-binding.csv, layer ≥ {LATE_LAYER_MIN}): {late_heads}
Result: {n_qual}/{n_late} late-layer top binders sink-dominated (BOS ≥ {SINK_THRESHOLD}) or structural (pos-1 ≥ {STRUCTURAL_THRESHOLD}).
Frozen bar: ≥ 50% ⇒ CONFIRMED; >0 <50% ⇒ PARTIAL; 0 ⇒ NOT CONFIRMED.

Per-head table: d1_late_head_characterization.csv
Paper consequences per outcome mapping in the pre-registration. All branches ship.
Instrument: transformer-lens==2.17.0, one characterization forward pass.
"""
(OUT_DIR / "VERDICT.md").write_text(verdict_md)
print(f"\nWritten: {OUT_DIR / 'd1_late_head_characterization.csv'}")
print(f"Written: {OUT_DIR / 'VERDICT.md'}")


Written: /content/results/d1_bos_diagnostic/d1_late_head_characterization.csv
Written: /content/results/d1_bos_diagnostic/VERDICT.md


## After the run
1. Commit results/d1_bos_diagnostic/ (CSV + VERDICT.md).
2. Bring the verdict to the thread — text consequences execute per the
   pre-registration's outcome mapping, Trisha ratifying as always.